<a href="https://colab.research.google.com/github/George7531/Portfolio_Management/blob/main/Group_19296_MScFE_652_Portfolio_Management_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Code Book:**

##Group Work Project # 2
###GROUP NUMBER: 19296
####Group members:

- Duy Vo
- George Immanuel
- Sing Chun Lee


#### Import Required Packages

In [ ]:
# This block import all required python packages for running this notebook
import numpy as np                      # numpy
import matplotlib.pyplot as plt         # for plotting
import math                             # math lib
import pandas as pd                     # for storing data in the DataFrame
import pandas_datareader.data as web
from scipy.optimize import minimize     # for minimizing
import yfinance as yf                   # yfinance library

# Data Preperation and Description

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_datareader.data as web

# Define parameters
tickers = ['TSLA', 'WMT', 'BAC', 'GS', 'LLY', 'MRK', 'GOOG', 'META', 'AAPL', 'XOM']
BENCHMARK = "^GSPC"
in_sample_start = '2023-09-01'
in_sample_end = '2025-09-30'
out_sample_start = '2025-10-01'
out_sample_end = '2025-12-31'

# 1. Download using yfinance
# Using auto_adjust=False
raw_data = yf.download(
    tickers + [BENCHMARK],
    start='2023-08-25', # For initial return
    end='2026-01-01', # the end is exclusive, we should use 2026-01-01 so 2025-12-31 is included
    auto_adjust=False
)

adj_close = raw_data['Close']

# Calculate simple daily percentage returns
daily_returns = adj_close.pct_change().dropna(how='all')

# Drop any trading days where any of the 10 assets have missing data (strict alignment)
daily_returns = daily_returns.dropna()

# In-Sample and Out-of-Sample subsets
returns_in_sample = daily_returns.loc[in_sample_start:in_sample_end]
returns_out_sample = daily_returns.loc[out_sample_start:out_sample_end]

print(f"In-Sample Observations: {len(returns_in_sample)} days")
print(f"Out-of-Sample Observations: {len(returns_out_sample)} days")

# 2. Download Fama-French 5-Factor Daily Data
ff5_daily = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2023-09-01', end='2026-01-01')[0]

# Convert from percentages to decimals
ff5_daily = ff5_daily / 100.0

# Partition factor data
ff_in_sample = ff5_daily.loc[in_sample_start:in_sample_end]
ff_out_sample = ff5_daily.loc[out_sample_start:out_sample_end]

print("\nFama-French Data Sample:")
print(pd.concat([ff_in_sample.head(2), ff_out_sample.tail(3)]))

[*********************100%***********************]  11 of 11 completed


In-Sample Observations: 521 days
Out-of-Sample Observations: 64 days

Fama-French Data Sample:
            Mkt-RF     SMB     HML     RMW     CMA      RF
Date                                                      
2023-09-01  0.0026  0.0085  0.0058  0.0000 -0.0022  0.0002
2023-09-05 -0.0054 -0.0157 -0.0087 -0.0030 -0.0048  0.0002
2025-12-29 -0.0041 -0.0015  0.0007  0.0032  0.0003  0.0002
2025-12-30 -0.0020 -0.0049  0.0029  0.0036  0.0014  0.0002
2025-12-31 -0.0076  0.0006 -0.0010 -0.0007 -0.0004  0.0002


# George's work:


# Step 4c: Detoning

Financial assets are often correlated because they are exposed to some common systematic factors, it could be recession or recent policy changes or whatever that makes all assets in the market to behave like they are all same. A simple example would be the oil syndicate deciding to change the price of fuel, since majority of vehicles use petrol/diesel for locomotion, it will simultaneously affect the prices of all assets from sandals we wear, to colab we use to write this piece down, to colors they use to make floral patterns on their porches(Indian cultural custom) likewise in times of crisis all assets from real estates, to IT stocks, to agricultural companies are affected and heavily correlated and nose dive regardless of their locations and geography in the globe.

How do we deal with this?
When we calculate the correlation matrix of asset returns, the largest eigenvalue and its corresponding eigenvector often capture this common market behaviour. This is called as the market mode. But the things is the market factor that is common among assets would overstate or misreport the actual specific dependency or correlation between companies for example lets say Nvidia and AMD dealing with semiconductors and chips they have some dependency between them that is prior to including the common market factor such as oil price was 0.5 correlation but after including the market factor it might elevate to 0.91 or some other bizzare number that obscures the actual dependency between things we measure. This is where Detoning helps.

Detoning addresses this problem by removing the contribution of the dominant market eigencomponent from the correlation matrix.This allows the remaining correlation structure to reflect more of the relative relationships between assets rather than their common market exposure (López de Prado, 2020).

The steps to detoning are way simpler with statistical packages such as numpy or scipy, I just want to state here the steps in words.

* step1: Take the correlation matrix of our portfolio. Constraint is this correlation matrix must be postive semi-definite meaning its eigen values must be greater than or equal to 0.

* step2: Take eigen decomposition to get eigen vectors and their eigen values$ Correl = V \Lambda V^T $.  Arrange the eigen values and corresponding eigen vectors in descending order to identify the largest contributer(s) as front guardians ($\lambda_1 \geq \lambda_2 \geq \dots \geq \lambda_N$). The largest eigen value ($ \lambda_1$) and its vector($v_1$) typically represent major market factor's contribution.

* step3: In order to remove the effect of this common factor from our correlation matrix, we have to subtract the original covariance matrix from this largest(er) eigen value and vector such as: $ \hat{correl} = correl - \lambda_1 v_1 v_1^T $, if we want to remove q major factors then we will have to do: $$\hat{Correl} = Correl - \sum_{i=1}^{q} \lambda_i v_i v_i^\top$$

* step4: Then we have to normalize this $ \hat{correl} $ to make it have 1 as its diagonal entries because when we did the transformation(by removing the major contributers by subtraction) the diagonal entries may have gotten altered, so longer representative of self-correlation, so we do this by taking the diagonal elements of correl and then take inverse square root of each entry like this:
$$D^{-1/2} = \begin{pmatrix} \frac{1}{\sqrt{\widetilde{C}_{11}}} & 0 & \cdots & 0 \\  0 & \frac{1}{\sqrt{\widetilde{C}_{22}}} & \cdots & 0 \\  \vdots & \vdots & \ddots & \vdots \\  0 & 0 & \cdots & \frac{1}{\sqrt{\widetilde{C}_{NN}}} \end{pmatrix}$$

to finally get our detoned correlated matrix:


$$ Correl_{\text{detoned}} = D^{-1/2} \hat{Correl} D^{-1/2} $$

This detoning technique can be super useful in hierarchical risk clustering or for risk seperation especially during times of crisis when assets in our portfolio would exhibit higher and closer correlations like 0.91,0.93 and so on.. during which time we can remove the common market factor(s) contribution and find how each individual asset are correlated in the absence of that common factor.

